In [8]:

from datetime import datetime, timedelta
import pandas as pd
from dateutil.relativedelta import relativedelta


def _advance(date, n, unit):
    """Move `date` forward by n units ('D' or 'M'). n may be negative."""
    if unit == 'M':
        return date + relativedelta(months=n)
    elif unit == 'D':
        return date + timedelta(days=n)
    raise ValueError("unit must be 'D' or 'M'")


def _sliding_windows(start_date_str, end_date_str, icl, ocl, unit, stride):
    """Shared sliding-window logic. Returns a list of dicts with generic keys."""
    global_start = datetime.strptime(start_date_str, '%Y-%m-%d')
    global_end = datetime.strptime(end_date_str, '%Y-%m-%d')

    current_input_start = global_start
    window_idx = 0
    rows = []

    while True:
        input_end = _advance(current_input_start, icl - 1, unit)
        output_start = _advance(input_end, 1, unit)
        output_end = _advance(current_input_start, icl + ocl - 1, unit)

        if output_end > global_end:
            break

        rows.append({
            "WINDOW": window_idx,
            "INPUT_START": current_input_start.strftime('%Y-%m-%d'),
            "INPUT_END": input_end.strftime('%Y-%m-%d'),
            "OUTPUT_START": output_start.strftime('%Y-%m-%d'),
            "OUTPUT_END": output_end.strftime('%Y-%m-%d'),
        })

        current_input_start = _advance(current_input_start, stride, unit)
        window_idx += 1

    if not rows:
        needed = icl + ocl
        print(f"No windows fit: {start_date_str} to {end_date_str} is shorter than "
              f"icl+ocl={needed} {unit}. Widen the range, or shrink icl/ocl.")

    return rows


def get_train_windows(train_start_date, train_end_date, icl, ocl, unit, stride=1):
    """
    All TFT training windows sliding between train_start_date and train_end_date.

    Returns
    -------
    pd.DataFrame: WINDOW, TRAIN_START_INPUT, TRAIN_END_INPUT, OUTPUT_START, OUTPUT_END
    """
    rows = _sliding_windows(train_start_date, train_end_date, icl, ocl, unit, stride)
    df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame(columns=["WINDOW", "TRAIN_START_INPUT", "TRAIN_END_INPUT",
                                      "OUTPUT_START", "OUTPUT_END"])
    return df.rename(columns={"INPUT_START": "TRAIN_START_INPUT",
                               "INPUT_END": "TRAIN_END_INPUT"})


def get_val_windows(val_start_date, val_end_date, icl, ocl, unit, stride=1):
    """
    All TFT validation windows sliding between val_start_date and val_end_date.

    Returns
    -------
    pd.DataFrame: WINDOW, VAL_START_INPUT, VAL_END_INPUT, OUTPUT_START, OUTPUT_END
    """
    rows = _sliding_windows(val_start_date, val_end_date, icl, ocl, unit, stride)
    df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame(columns=["WINDOW", "VAL_START_INPUT", "VAL_END_INPUT",
                                      "OUTPUT_START", "OUTPUT_END"])
    return df.rename(columns={"INPUT_START": "VAL_START_INPUT",
                               "INPUT_END": "VAL_END_INPUT"})


In [10]:
import pandas as pd

def _generate_sliding_windows(start_date, end_date, icl, ocl, stride, prefix, freq='D'):
    """
    Generate sliding window bounds for daily or monthly series.
    - freq: 'D' for daily data, 'M' for monthly data
    """
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    # Select offset unit based on frequency
    freq_upper = freq.upper()
    if freq_upper in ['D', 'DAILY']:
        step = lambda x: pd.DateOffset(days=x)
    elif freq_upper in ['M', 'MONTHLY']:
        step = lambda x: pd.DateOffset(months=x)
    else:
        raise ValueError("freq must be either 'D' (daily) or 'M' (monthly)")

    records = []
    window_idx = 1
    current_input_start = start

    while True:
        # Calculate window boundary dates using frequency offsets
        current_input_end = current_input_start + step(icl - 1)
        current_output_start = current_input_end + step(1)
        current_output_end = current_output_start + step(ocl - 1)

        # Stop condition: total window exceeds the end date
        if current_output_end > end:
            break

        records.append({
            'WINDOW': f"{prefix}_WINDOW_{window_idx}",
            f'{prefix}_START_INPUT': current_input_start.strftime('%Y-%m-%d'),
            f'{prefix}_END_INPUT': current_input_end.strftime('%Y-%m-%d'),
            'OUTPUT_START': current_output_start.strftime('%Y-%m-%d'),
            'OUTPUT_END': current_output_end.strftime('%Y-%m-%d')
        })

        window_idx += 1
        current_input_start += step(stride)

    return pd.DataFrame(records)


def get_train_windows(train_start_date, train_end_date, icl, ocl, stride, freq='D'):
    return _generate_sliding_windows(
        train_start_date, train_end_date, icl, ocl, stride, prefix='TRAIN', freq=freq
    )


def get_val_windows(val_start_date, val_end_date, icl, ocl, stride, freq='D'):
    return _generate_sliding_windows(
        val_start_date, val_end_date, icl, ocl, stride, prefix='VAL', freq=freq
    )

In [35]:
get_train_windows('2023-04-01','2026-04-30',365,100,1,freq='D')

,WINDOW,TRAIN_START_INPUT,TRAIN_END_INPUT,OUTPUT_START,OUTPUT_END
0,TRAIN_WINDOW_1,2023-04-01,2024-03-30,2024-03-31,2024-07-08
1,TRAIN_WINDOW_2,2023-04-02,2024-03-31,2024-04-01,2024-07-09
2,TRAIN_WINDOW_3,2023-04-03,2024-04-01,2024-04-02,2024-07-10
3,TRAIN_WINDOW_4,2023-04-04,2024-04-02,2024-04-03,2024-07-11
4,TRAIN_WINDOW_5,2023-04-05,2024-04-03,2024-04-04,2024-07-12
...,...,...,...,...,...
657,TRAIN_WINDOW_658,2025-01-17,2026-01-16,2026-01-17,2026-04-26
658,TRAIN_WINDOW_659,2025-01-18,2026-01-17,2026-01-18,2026-04-27
659,TRAIN_WINDOW_660,2025-01-19,2026-01-18,2026-01-19,2026-04-28
660,TRAIN_WINDOW_661,2025-01-20,2026-01-19,2026-01-20,2026-04-29


In [25]:
pd.to_datetime('2026-07-31') + relativedelta(days=-(365+184))

Timestamp('2025-01-28 00:00:00')

In [34]:
delta = pd.to_datetime('2026-12-10')-pd.to_datetime('2026-09-01')
delta.days

100

In [36]:
pd.to_datetime('2026-07-31') + relativedelta(days=-(365+100))

Timestamp('2025-04-22 00:00:00')

In [38]:
get_val_windows('2025-04-22','2026-07-31',365,100,1,freq='D')

,WINDOW,VAL_START_INPUT,VAL_END_INPUT,OUTPUT_START,OUTPUT_END
0,VAL_WINDOW_1,2025-04-23,2026-04-22,2026-04-23,2026-07-31
